# IEEE-CIS Fraud Detection — Neural Network (Best Basic Model)

## Architecture Overview

| Component | Detail |
|---|---|
| Input | Individual embedding per categorical column + scaled numerical features |
| Hidden layers | 3 blocks: H → H/2 → H/4, each: `Linear → BatchNorm → GELU` |
| Weight init | Xavier uniform (weights), zero (biases) |
| Optimiser | Lookahead(AdamW) |
| Loss | BCEWithLogitsLoss + `pos_weight` (class-imbalance correction) |
| Regularisation | L2 via AdamW `weight_decay` |
| Scheduler | ReduceLROnPlateau (mode=max, tracks val PR-AUC) |
| HP search | Bayesian optimisation via Optuna TPE sampler (30 trials, 8 warm-up epochs each) |
| Gradient clipping | `max_norm=1.0` |
| Primary metric | PR-AUC |
| Threshold | Best F1 on test Precision-Recall curve |
| Checkpointing | `best.pt` (highest val PR-AUC) and `last.pt` (most recent epoch) |

## Notebook Structure
1. Installs
2. Imports & Global Seeding
3. Training Components (`Lookahead`)
4. Dataset & Model (`FraudDataset`, `FraudNet`)
5. Training Pipeline (epoch loop, checkpointing, logging)
6. Hyperparameter Tuning (Optuna Bayesian search)
7. Evaluation & Visualisation (PR-AUC, confusion matrix, PR curve)
8. Main Entrypoint


## 1. Installs

Install all required packages. Run once per environment.

In [ ]:
pip install numpy pandas scikit-learn matplotlib seaborn optuna torch torchvision torchaudio tqdm joblib


## 2. Imports and Global Seeding

All random seeds are fixed globally **before** any other operation to ensure full reproducibility
across Python, NumPy, and PyTorch (CPU, CUDA, and MPS backends).
The `SEED` constant is used consistently throughout the notebook.


In [ ]:
import os
import json
import pickle
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    recall_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED: int = 42

def set_seed(seed: int = SEED) -> None:
    """Fix all random seeds for full reproducibility across backends."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.mps.is_available():
        try:
            torch.mps.manual_seed(seed)
        except AttributeError:
            pass  # older PyTorch builds may not expose mps.manual_seed

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Device selection: prefer MPS (Apple Silicon) → CUDA → CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device : {device}")
print(f"Global seed  : {SEED}")


## 3. Training Components


### 3.1 `Lookahead`
Lookahead optimiser wrapper [(Zhang et al., 2019)](https://arxiv.org/abs/1907.08610) maintains
a set of slow weights that are periodically updated toward the fast weights produced by the
inner optimiser (AdamW). This reduces sensitivity to the learning rate and improves convergence
stability.


In [ ]:
class Lookahead(torch.optim.Optimizer):
    """Lookahead optimiser wrapper (Zhang et al., 2019).

    Wraps any base optimiser and maintains a parallel set of slow weights.
    Every `k` inner steps, the slow weights are interpolated toward the fast
    weights by a factor of `alpha`, then the fast weights are reset to the
    slow weights. This improves stability and reduces variance.

    Args:
        base_optimizer: Inner optimiser (e.g., AdamW) to wrap.
        k: Number of inner (fast) steps before a slow-weight update.
           Tuned by Optuna from {5, 10}.
        alpha: Interpolation coefficient for the slow-weight update.
               Tuned by Optuna in range [0.3, 0.8].
    """

    def __init__(self, base_optimizer: torch.optim.Optimizer, k: int = 5, alpha: float = 0.5) -> None:
        self.base_optimizer = base_optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = base_optimizer.param_groups
        self.state = base_optimizer.state
        self._step_counter = 0
        self.slow_weights = [
            [p.clone().detach() for p in group["params"]]
            for group in self.param_groups
        ]

    def step(self, closure=None):
        """Perform one inner optimiser step; update slow weights every k steps."""
        loss = self.base_optimizer.step(closure)
        self._step_counter += 1

        if self._step_counter % self.k == 0:
            for group, slow_group in zip(self.param_groups, self.slow_weights):
                for p, slow_p in zip(group["params"], slow_group):
                    if p.grad is None:
                        continue
                    slow_p.data.add_(self.alpha * (p.data - slow_p.data))
                    p.data.copy_(slow_p.data)

        return loss

    def zero_grad(self, set_to_none: bool = False) -> None:
        """Delegate gradient zeroing to the base optimiser."""
        self.base_optimizer.zero_grad(set_to_none=set_to_none)

    def state_dict(self) -> dict:
        """Return serialisable state (base optimiser + slow weights)."""
        return {
            "base_optimizer": self.base_optimizer.state_dict(),
            "step_counter": self._step_counter,
            "slow_weights": [[p.cpu() for p in g] for g in self.slow_weights],
        }

    def load_state_dict(self, state: dict) -> None:
        """Restore state and move slow weights to the correct device."""
        self.base_optimizer.load_state_dict(state["base_optimizer"])
        self._step_counter = state["step_counter"]
        device = next(iter(self.base_optimizer.param_groups[0]["params"])).device
        self.slow_weights = [
            [p.to(device) for p in g] for g in state["slow_weights"]
        ]


## 4. Dataset and Model


### 4.1 `FraudDataset`
A lightweight PyTorch `Dataset` that stores categorical features as `long` tensors
(required by `nn.Embedding`) and numerical features as `float32` tensors.


In [ ]:
class FraudDataset(Dataset):
    """PyTorch Dataset for tabular fraud detection data.

    Stores categorical columns as long (int64) tensors for embedding lookup
    and numerical columns as float32 tensors for direct network input.

    Args:
        df: DataFrame containing feature and target columns.
        cat_cols: List of categorical column names (integer-encoded).
        num_cols: List of numerical column names (pre-scaled).
        target: Name of the binary target column.
    """

    def __init__(self, df: pd.DataFrame, cat_cols: list, num_cols: list, target: str) -> None:
        self.cat = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.y   = torch.tensor(df[target].values,   dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        """Return (categorical_features, numerical_features, label) for one sample."""
        return self.cat[idx], self.num[idx], self.y[idx]


### 4.2 `FraudNet`
Tabular neural network combining:
- **Per-column embeddings** — each categorical column gets its own `nn.Embedding`;
  embedding dimension is `min(50, (vocab_size + 1) // 2)` (a standard heuristic).
- **3-block MLP** — dimensions H → H/2 → H/4, each block: `Linear → BatchNorm → GELU`.
- **Xavier uniform** weight initialisation with zero biases for stable training.

> **Note vs Final Model:** This baseline uses `BCEWithLogitsLoss + pos_weight` instead of
> Focal Loss, and omits Dropout from hidden blocks. These are upgraded in the Final Model.


In [ ]:
class FraudNet(nn.Module):
    """Tabular neural network with per-column embeddings for fraud detection.

    Architecture:
      1. Each categorical column is passed through its own ``nn.Embedding``.
      2. All embeddings are concatenated with the scaled numerical features.
      3. The combined vector passes through three hidden blocks:
         ``Linear → BatchNorm1d → GELU``
         with dimensions H → H/2 → max(H/4, 32).
      4. A final linear layer outputs a single logit (binary classification).

    Weight initialisation:
      Xavier uniform for all ``nn.Linear`` weights; zero for all biases.

    Args:
        cat_cols: Ordered list of categorical column names.
        vocab_sizes: Mapping from column name to vocabulary size
                     (number of unique integer codes + 1 for unseen).
        num_dim: Number of numerical input features.
        hidden_dim: Width of the first hidden layer (H). Subsequent layers
                    are H/2 and max(H/4, 32). Tuned by Optuna from {128, 256, 512}.
    """

    def __init__(
        self,
        cat_cols: list,
        vocab_sizes: dict,
        num_dim: int,
        hidden_dim: int,
    ) -> None:
        super().__init__()

        # Per-column embedding tables; dim = min(50, (vocab + 1) // 2)
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], min(50, (vocab_sizes[col] + 1) // 2))
            for col in cat_cols
        ])

        emb_dim   = sum(min(50, (vocab_sizes[col] + 1) // 2) for col in cat_cols)
        input_dim = emb_dim + num_dim

        # Hidden layer widths
        h1 = hidden_dim
        h2 = h1 // 2
        h3 = max(h2 // 2, 32)  # floor at 32 to avoid collapse in small configs

        def _block(in_f: int, out_f: int) -> nn.Sequential:
            """One hidden block: Linear → BatchNorm → GELU (no Dropout in this baseline)."""
            return nn.Sequential(
                nn.Linear(in_f, out_f),
                nn.BatchNorm1d(out_f),
                nn.GELU(),
            )

        self.net = nn.Sequential(
            _block(input_dim, h1),
            _block(h1, h2),
            _block(h2, h3),
            nn.Linear(h3, 1),   # output logit
        )

        self._xavier_init()

    def _xavier_init(self) -> None:
        """Apply Xavier uniform initialisation to all Linear layers."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, cat: torch.Tensor, num: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            cat: Integer-coded categorical features, shape (N, C).
            num: Scaled numerical features, shape (N, D).

        Returns:
            Raw logits, shape (N,).
        """
        embs = [e(cat[:, i]) for i, e in enumerate(self.embeddings)]
        x = torch.cat(embs + [num], dim=1)
        return self.net(x).squeeze(1)

    @torch.no_grad()
    def predict_proba(self, cat: torch.Tensor, num: torch.Tensor) -> torch.Tensor:
        """Return sigmoid probabilities (no gradient tracked).

        Args:
            cat: Integer-coded categorical features, shape (N, C).
            num: Scaled numerical features, shape (N, D).

        Returns:
            Predicted probabilities in [0, 1], shape (N,).
        """
        return torch.sigmoid(self.forward(cat, num))


## 5. Training Pipeline

The training pipeline consists of four functions:

| Function | Purpose |
|---|---|
| `get_pos_weight` | Compute neg/pos ratio as a scalar tensor for `BCEWithLogitsLoss` |
| `train_epoch` | Single forward/backward pass over the training DataLoader with gradient clipping |
| `save_checkpoint` / `load_checkpoint` | Persist and restore model + optimiser state with full metadata |
| `run_training` | Full training loop: epoch iteration, validation, scheduling, checkpointing, Optuna pruning support |


In [ ]:
def get_pos_weight(y_series: pd.Series, device: torch.device) -> torch.Tensor:
    """Compute the negative-to-positive class ratio for BCEWithLogitsLoss.

    A higher pos_weight upweights the positive (fraud) class in the loss,
    compensating for the severe class imbalance in fraud detection datasets.

    Args:
        y_series: Binary target column from the training DataFrame.
        device: Compute device to place the tensor on.

    Returns:
        Scalar tensor equal to n_negatives / n_positives.
    """
    n_neg = (y_series == 0).sum()
    n_pos = (y_series == 1).sum()
    return torch.tensor(n_neg / n_pos, dtype=torch.float32).to(device)


def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """Run one full training epoch over the provided DataLoader.

    Applies gradient clipping (max_norm=1.0) before each parameter update
    to prevent exploding gradients.

    Args:
        model: The neural network to train.
        loader: Training DataLoader (yields cat, num, y batches).
        optimizer: Lookahead(AdamW) optimiser instance.
        criterion: BCEWithLogitsLoss instance.
        device: Compute device (cpu / cuda / mps).

    Returns:
        Sample-weighted mean training loss for the epoch.
    """
    model.train()
    total_loss = 0.0

    for cat, num, y in loader:
        cat, num, y = cat.to(device), num.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(cat, num), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)   # accumulate sample-weighted loss

    return total_loss / len(loader.dataset)


def save_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    val_pr: float,
    config: dict,
    metadata: dict,
) -> None:
    """Save a full training checkpoint to disk.

    Saves model weights, optimiser state, hyperparameter config, and
    column metadata so the checkpoint is fully self-contained for inference.

    Args:
        path: Destination file path (e.g., ``runs/fraud_nn/best.pt``).
        model: Trained FraudNet instance.
        optimizer: Lookahead optimiser (state includes slow weights).
        epoch: Current epoch number (1-indexed).
        val_pr: Validation PR-AUC at this checkpoint.
        config: Hyperparameter dict used for this training run.
        metadata: Column metadata dict (cat_cols, num_cols, vocab_sizes).
    """
    torch.save({
        "epoch":            epoch,
        "val_pr_auc":       val_pr,
        "config":           config,
        "model_state_dict": model.state_dict(),
        "optimizer_state":  optimizer.state_dict(),
        "cat_cols":         metadata["cat_cols"],
        "num_cols":         metadata["num_cols"],
        "vocab_sizes":      metadata["vocab_sizes"],
    }, path)
    print(f"    Saved -> {path}")


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer = None,
    device: str = "cpu",
) -> dict:
    """Load model (and optionally optimiser) state from a checkpoint.

    Args:
        path: Path to a ``.pt`` checkpoint file produced by ``save_checkpoint``.
        model: FraudNet instance whose weights will be overwritten in-place.
        optimizer: If provided, optimiser state is also restored.
        device: Device string for ``torch.load`` map_location.

    Returns:
        The raw checkpoint dict (contains epoch, val_pr_auc, config, etc.).
    """
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt


def run_training(
    config: dict,
    train_ds: Dataset,
    val_ds: Dataset,
    vocab_sizes: dict,
    cat_cols: list,
    num_cols: list,
    pos_weight: torch.Tensor,
    metadata: dict,
    device: torch.device,
    run_dir: str = None,
    n_epochs: int = 40,
    trial=None,
) -> tuple:
    """Full training loop with validation, checkpointing, and Optuna integration.

    Builds the DataLoaders, constructs the model and optimiser, then iterates
    for ``n_epochs`` epochs. The best checkpoint (by val PR-AUC) is saved to
    ``run_dir/best.pt``; the most recent checkpoint is saved to ``run_dir/last.pt``.

    If ``trial`` is provided (Optuna integration), intermediate values are
    reported after each epoch and the trial is pruned if the MedianPruner
    determines it is unpromising.

    Args:
        config: Hyperparameter dict (hidden_dim, lr, weight_decay, batch_size,
                la_k, la_alpha, sched_patience, sched_factor).
        train_ds: Training FraudDataset.
        val_ds: Validation FraudDataset.
        vocab_sizes: Mapping from categorical column name to vocabulary size.
        cat_cols: Ordered list of categorical column names.
        num_cols: List of numerical column names.
        pos_weight: Scalar tensor (neg/pos ratio) for BCEWithLogitsLoss.
        metadata: Full column metadata dict (passed through to checkpoints).
        device: Compute device.
        run_dir: Directory for saving checkpoints and training log.
                 If None, no files are written (used during HP search).
        n_epochs: Number of training epochs.
        trial: Optional ``optuna.Trial`` for pruning and reporting.

    Returns:
        Tuple of (best_val_pr_auc, best_state_dict, history_dict).
        ``best_state_dict`` contains CPU-moved parameter tensors.
        ``history_dict`` has keys ``'train_loss'`` and ``'val_pr'`` (lists).

    Raises:
        optuna.exceptions.TrialPruned: If the Optuna pruner triggers.
    """
    loader_tr = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=False,
    )
    loader_va = DataLoader(
        val_ds,
        batch_size=config["batch_size"],
        shuffle=False,   # no shuffle needed for evaluation
        num_workers=0,
        pin_memory=False,
    )

    # Build model
    model = FraudNet(
        cat_cols=cat_cols,
        vocab_sizes=vocab_sizes,
        num_dim=len(num_cols),
        hidden_dim=config["hidden_dim"],
    ).to(device)

    # Build optimiser: Lookahead wraps AdamW with L2 regularisation
    inner_opt = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    optimizer = Lookahead(inner_opt, k=config["la_k"], alpha=config["la_alpha"])

    # LR scheduler monitors val PR-AUC; reduces LR if no improvement for `patience` epochs
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        inner_opt,
        mode="max",
        patience=config["sched_patience"],
        factor=config["sched_factor"],
    )

    # BCEWithLogitsLoss with pos_weight to handle class imbalance
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    history = {"train_loss": [], "val_pr": []}
    best_state = None
    best_val_pr_auc = 0.0

    print(f"  {'Epoch':>6}  {'Loss':>8}  {'Val PR-AUC':>10}  {'Time':>7}")
    print(f"  {'-'*6}  {'-'*8}  {'-'*10}  {'-'*7}")

    epoch_log_rows = []

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()

        train_loss = train_epoch(model, loader_tr, optimizer, criterion, device)
        val_pr     = eval_pr_auc(model, loader_va, device)
        scheduler.step(val_pr)   # update LR based on val PR-AUC

        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_pr"].append(val_pr)

        # Save best checkpoint whenever val PR-AUC improves
        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if run_dir:
                save_checkpoint(
                    os.path.join(run_dir, "best.pt"),
                    model, optimizer, epoch, val_pr, config, metadata,
                )

        # Always save the latest checkpoint
        if run_dir:
            save_checkpoint(
                os.path.join(run_dir, "last.pt"),
                model, optimizer, epoch, val_pr, config, metadata,
            )

        print(f"  {epoch:>6}/{n_epochs}  {train_loss:>8.4f}  {val_pr:>10.4f}  {elapsed:>6.1f}s")

        epoch_log_rows.append({
            "epoch":      epoch,
            "train_loss": train_loss,
            "val_pr":     val_pr,
            "time_s":     round(elapsed, 1),
        })

        # Optuna integration: report intermediate value and check for pruning
        if trial is not None:
            trial.report(val_pr, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    # Persist per-epoch training log to CSV
    if run_dir:
        pd.DataFrame(epoch_log_rows).to_csv(
            os.path.join(run_dir, "training_log.csv"),
            index=False,
        )

    return best_val_pr_auc, best_state, history


## 6. Hyperparameter Tuning

Bayesian optimisation via Optuna's TPE (Tree-structured Parzen Estimator) sampler.
The study is seeded (`TPESampler(seed=SEED)`) for reproducibility.

**Search space:**

| Hyperparameter | Type | Range / Options |
|---|---|---|
| `hidden_dim` | categorical | {128, 256, 512} |
| `lr` | log-uniform | [1e-4, 1e-2] |
| `weight_decay` | log-uniform | [1e-6, 1e-2] |
| `batch_size` | categorical | {512, 1024, 4096} |
| `la_k` | categorical | {5, 10} |
| `la_alpha` | uniform | [0.3, 0.8] |
| `sched_patience` | categorical | {3, 5} |
| `sched_factor` | uniform | [0.3, 0.7] |

Unpromising trials are pruned after 3 warm-up epochs using the **MedianPruner**.

> **Note vs Final Model:** The Final Model additionally tunes `dropout`, `alpha` (FocalLoss), and `gamma` (FocalLoss).


In [ ]:
def bayesian_search(
    train_ds: Dataset,
    val_ds: Dataset,
    vocab_sizes: dict,
    cat_cols: list,
    num_cols: list,
    pos_weight: torch.Tensor,
    metadata: dict,
    device: torch.device,
    n_trials: int = 30,
    n_epochs_tune: int = 8,
) -> tuple:
    """Run Bayesian hyperparameter search using Optuna TPE.

    Each trial trains for ``n_epochs_tune`` epochs (short-horizon proxy).
    The study is seeded with the global ``SEED`` for reproducibility.
    Unpromising trials are pruned via MedianPruner (3 warm-up steps).

    Args:
        train_ds: Training FraudDataset.
        val_ds: Validation FraudDataset.
        vocab_sizes: Vocabulary sizes for each categorical column.
        cat_cols: Ordered list of categorical column names.
        num_cols: List of numerical column names.
        pos_weight: Class-imbalance correction tensor for BCEWithLogitsLoss.
        metadata: Column metadata dict (passed to run_training).
        device: Compute device.
        n_trials: Number of Optuna trials. Default 30.
        n_epochs_tune: Epochs per trial. Default 8.

    Returns:
        Tuple of (best_params_dict, optuna_study).
    """

    def objective(trial: optuna.Trial) -> float:
        """Optuna objective: returns val PR-AUC for a sampled hyperparameter config."""
        config = {
            "hidden_dim":     trial.suggest_categorical("hidden_dim",     [128, 256, 512]),
            "lr":             trial.suggest_float("lr",                   1e-4, 1e-2, log=True),
            "weight_decay":   trial.suggest_float("weight_decay",         1e-6, 1e-2, log=True),
            "batch_size":     trial.suggest_categorical("batch_size",     [512, 1024, 4096]),
            "la_k":           trial.suggest_categorical("la_k",           [5, 10]),
            "la_alpha":       trial.suggest_float("la_alpha",             0.3,  0.8),
            "sched_patience": trial.suggest_categorical("sched_patience", [3, 5]),
            "sched_factor":   trial.suggest_float("sched_factor",         0.3,  0.7),
        }

        val_pr, _, _ = run_training(
            config, train_ds, val_ds, vocab_sizes, cat_cols, num_cols,
            pos_weight, metadata, device,
            run_dir=None,          # no checkpoints during search
            n_epochs=n_epochs_tune,
            trial=trial,
        )
        return val_pr

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=SEED),          # seeded for reproducibility
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        callbacks=[
            lambda s, t: print(
                f"  [Trial {t.number:>2}] val_PR-AUC="
                f"{(t.value if t.value is not None else float('nan')):.4f}"
                f"  params={t.params}"
            )
        ],
    )

    print(f"\nBest trial: val_PR-AUC={study.best_value:.4f}")
    print(f"Best params: {study.best_params}")
    return study.best_params, study


## 7. Evaluation and Visualisation

| Function | Purpose |
|---|---|
| `eval_pr_auc` | Compute PR-AUC (average precision) over a DataLoader |
| `evaluate_metrics` | Convenience wrapper for `eval_pr_auc` (infers device from model) |
| `get_predictions` | Collect all ground-truth labels and predicted probabilities |
| `plot_confusion_matrix` | Plot raw-count and row-normalised confusion matrices; save classification report |

> **Note vs Final Model:** The Final Model additionally provides `find_best_fbeta_threshold`
> which selects the decision threshold on the **validation set** using F-β (β=2.0) to
> prioritise recall. This baseline selects the threshold on the test PR curve directly (data leakage risk).


In [ ]:
@torch.no_grad()
def eval_pr_auc(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    """Compute PR-AUC (average precision score) over a DataLoader.

    Runs the model in evaluation mode.

    Args:
        model: Trained FraudNet in any mode; switched to eval internally.
        loader: DataLoader (yields cat, num, y batches).
        device: Compute device.

    Returns:
        Scalar PR-AUC (average precision) in [0, 1].
    """
    model.eval()
    all_p, all_y = [], []

    for cat, num, y in loader:
        cat, num = cat.to(device), num.to(device)
        all_p.extend(model.predict_proba(cat, num).cpu().numpy())
        all_y.extend(y.numpy())

    return average_precision_score(np.array(all_y), np.array(all_p))


def evaluate_metrics(model: nn.Module, loader: DataLoader) -> float:
    """Convenience wrapper: compute PR-AUC, inferring device from model parameters.

    Args:
        model: Trained FraudNet.
        loader: DataLoader to evaluate on.

    Returns:
        PR-AUC scalar.
    """
    return eval_pr_auc(model, loader, next(model.parameters()).device)


@torch.no_grad()
def get_predictions(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device = None,
) -> tuple:
    """Collect all ground-truth labels and predicted probabilities.

    Args:
        model: Trained FraudNet.
        loader: DataLoader to run inference on.
        device: Compute device; inferred from model if None.

    Returns:
        Tuple of ``(y_true, y_proba)`` as NumPy arrays of shape (N,).
    """
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    all_p, all_y = [], []

    for cat, num, y in loader:
        cat, num = cat.to(device), num.to(device)
        all_p.extend(model.predict_proba(cat, num).cpu().numpy())
        all_y.extend(y.numpy())

    return np.array(all_y), np.array(all_p)


def plot_confusion_matrix(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float,
    run_dir: str,
) -> None:
    """Plot raw-count and row-normalised confusion matrices; save classification report.

    Both the confusion matrix PNG and the classification report TXT are written
    to ``run_dir``.

    Args:
        y_true: Ground-truth binary labels, shape (N,).
        y_proba: Predicted probabilities in [0, 1], shape (N,).
        threshold: Decision threshold (best F1 on test PR curve).
        run_dir: Directory where outputs are saved.
    """
    y_pred = (y_proba >= threshold).astype(int)
    cm     = confusion_matrix(y_true, y_pred)
    labels = ["Non-Fraud", "Fraud"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: raw counts
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
        xticklabels=[f"Pred {l}" for l in labels],
        yticklabels=[f"True {l}" for l in labels],
    )
    axes[0].set_title(f"Confusion Matrix (thresh={threshold:.3f}) — Counts")

    # Right: row-normalised (true positive / false negative rates per class)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt=".2%", cmap="Blues", ax=axes[1],
        xticklabels=[f"Pred {l}" for l in labels],
        yticklabels=[f"True {l}" for l in labels],
    )
    axes[1].set_title(f"Confusion Matrix (thresh={threshold:.3f}) — Row-normalised")

    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "confusion_matrix.png"), dpi=150)
    plt.close()

    report = classification_report(y_true, y_pred, target_names=labels)
    print(report)

    with open(os.path.join(run_dir, "classification_report.txt"), "w") as f:
        f.write(f"Threshold: {threshold:.3f}\n\n{report}")


## 8. Main Entrypoint

The `main()` function orchestrates the full pipeline:

1. Load preprocessed train / val / test splits and column metadata.
2. Compute `pos_weight` from training labels for class-imbalance correction.
3. Run 30-trial Bayesian HP search (8 epochs each).
4. Retrain with the best config for 40 full epochs.
5. Select decision threshold on the **test** PR curve by maximising F1.
6. Evaluate on the test set; plot training curves, PR curve, and confusion matrix.
7. Save all artefacts to `runs/fraud_nn/`.

**Expected output files in `runs/fraud_nn/`:**
- `best.pt` — best model checkpoint (by val PR-AUC)
- `last.pt` — final epoch checkpoint
- `nn_model.pt` — standalone inference checkpoint with full metadata
- `training_log.csv` — per-epoch loss and val PR-AUC
- `hp_search_results.csv` — all Optuna trial results
- `final_metrics.json` — summary of all evaluation metrics
- `nn_training_curves.png` — loss and PR-AUC curves
- `pr_curve.png` — Precision-Recall curve on test set
- `confusion_matrix.png` — confusion matrix (counts + normalised)
- `classification_report.txt` — per-class classification report

> ⚠️ **Threshold leakage caveat:** This baseline selects the threshold on the test PR
> curve. The Final Model corrects this by selecting the threshold on the validation set
> using F-β (β=2.0) before any test-set evaluation.


In [ ]:
def main() -> None:
    """End-to-end pipeline: HP search → full training → evaluation → artefact saving."""

    # ── Configuration ─────────────────────────────────────────────────────────
    run_dir       = os.path.join("runs", "fraud_nn")
    n_epochs_full = 40   # epochs for the final full training run

    os.makedirs(run_dir, exist_ok=True)

    print(f"Device  : {device}")
    print(f"Seed    : {SEED}")
    print(f"Run dir : {run_dir}/")

    # ── 1. Load data ──────────────────────────────────────────────────────────
    print("\nLoading preprocessed data...")
    train_df = pd.read_csv("train.csv")
    val_df   = pd.read_csv("val.csv")
    test_df  = pd.read_csv("test.csv")

    with open("column_metadata.pkl", "rb") as f:
        metadata = pickle.load(f)

    target      = metadata["target"]
    cat_cols    = [c for c in metadata["cat_cols"]  if c in train_df.columns]
    num_cols    = [c for c in metadata["num_cols"]  if c in train_df.columns]
    vocab_sizes = metadata["vocab_sizes"]

    print(f"Categorical : {len(cat_cols)}")
    print(f"Numerical   : {len(num_cols)}")

    train_ds = FraudDataset(train_df, cat_cols, num_cols, target)
    val_ds   = FraudDataset(val_df,   cat_cols, num_cols, target)
    test_ds  = FraudDataset(test_df,  cat_cols, num_cols, target)

    # Compute class-imbalance weight from training labels only
    pos_weight = get_pos_weight(train_df[target], device)
    print(f"pos_weight  : {pos_weight.item():.2f}  (neg/pos ratio)")

    # ── 2. Bayesian hyperparameter search ─────────────────────────────────────
    print("\n" + "=" * 60)
    print("Bayesian Hyperparameter Search  (Optuna / TPE, 30 trials)")
    print("=" * 60)

    best_params, study = bayesian_search(
        train_ds, val_ds, vocab_sizes, cat_cols, num_cols,
        pos_weight, metadata, device,
        n_trials=30,
        n_epochs_tune=8,
    )

    # Save all trial results for reproducibility
    study.trials_dataframe().to_csv(
        os.path.join(run_dir, "hp_search_results.csv"),
        index=False,
    )

    # Assemble best config (explicit key list guards against unexpected extra params)
    best_config = {
        k: best_params[k]
        for k in [
            "hidden_dim", "lr", "weight_decay", "batch_size",
            "la_k", "la_alpha", "sched_patience", "sched_factor",
        ]
    }

    # ── 3. Full training with best config ─────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"Full training for {n_epochs_full} epochs with best config")
    print(f"{'=' * 60}\n")

    lr         = best_config["lr"]
    batch_size = best_config["batch_size"]

    loader_val  = DataLoader(val_ds,  batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    loader_test = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

    best_val_pr_auc, best_state, history = run_training(
        best_config, train_ds, val_ds, vocab_sizes,
        cat_cols, num_cols, pos_weight, metadata, device,
        run_dir=run_dir, n_epochs=n_epochs_full,
    )

    # Load best weights for evaluation
    model = FraudNet(
        cat_cols=cat_cols, vocab_sizes=vocab_sizes,
        num_dim=len(num_cols), hidden_dim=best_config["hidden_dim"],
    ).to(device)
    model.load_state_dict(best_state)

    val_pr  = evaluate_metrics(model, loader_val)
    test_pr = evaluate_metrics(model, loader_test)

    print(f"\n{'=' * 50}")
    print(f"  Checkpoint PR-AUC")
    print(f"{'=' * 50}")
    print(f"  Val  PR-AUC : {val_pr:.4f}")
    print(f"  Test PR-AUC : {test_pr:.4f}")

    # ── 4. Training curves ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.plot(history["train_loss"], label="Train Loss", color="steelblue")
    ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
    ax.set_title("Training Loss"); ax.legend()

    ax = axes[1]
    ax.plot(history["val_pr"], label="Val PR-AUC", color="green")
    ax.axhline(y=best_val_pr_auc, color="red", linestyle="--",
                label=f"Best = {best_val_pr_auc:.4f}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("PR-AUC")
    ax.set_title("Validation PR-AUC over Training"); ax.legend()

    cfg_label = f"lr={lr:.4f}, bs={batch_size}"
    plt.suptitle(f"Neural Network — IEEE Fraud Detection\n{cfg_label}", fontsize=11)
    plt.tight_layout()
    curves_path = os.path.join(run_dir, "nn_training_curves.png")
    plt.savefig(curves_path, dpi=150)
    plt.show()
    print(f"Training curves saved to {curves_path}")

    # ── 5. Threshold selection on TEST PR curve (best F1) ─────────────────────
    # NOTE: This selects threshold on the test set — see Final Model for the
    # correct approach of selecting on the validation set to avoid leakage.
    y_true, y_proba = get_predictions(model, loader_test)
    precision_pts, recall_pts, thresholds = precision_recall_curve(y_true, y_proba)

    f1_scores = (
        2 * precision_pts[:-1] * recall_pts[:-1]
        / (precision_pts[:-1] + recall_pts[:-1] + 1e-8)
    )
    best_idx       = np.argmax(f1_scores)
    best_thresh    = float(thresholds[best_idx])
    best_f1        = float(f1_scores[best_idx])
    best_recall    = float(recall_pts[best_idx])
    best_precision = float(precision_pts[best_idx])

    # ── 6. PR curve plot ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall_pts, precision_pts, color="steelblue", lw=2,
            label=f"PR curve (AUC = {test_pr:.4f})")
    ax.scatter(best_recall, best_precision, color="red", zorder=5, s=100,
                label=f"Best F1={best_f1:.4f}  thresh={best_thresh:.3f}")

    # ISO F1 contours for reference
    for f1_iso in [0.2, 0.4, 0.6, 0.8]:
        r_vals = np.linspace(0.01, 1.0, 300)
        p_vals = f1_iso * r_vals / (2 * r_vals - f1_iso + 1e-8)
        mask   = (p_vals >= 0) & (p_vals <= 1)
        ax.plot(r_vals[mask], p_vals[mask], "--", color="grey", lw=0.8, alpha=0.6)
        ax.annotate(f"F1={f1_iso}",
                    xy=(r_vals[mask][-1], p_vals[mask][-1]),
                    fontsize=7, color="grey")

    ax.set_xlabel("Recall", fontsize=12); ax.set_ylabel("Precision", fontsize=12)
    ax.set_title("Precision-Recall Curve — Test Set", fontsize=13)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.legend(fontsize=10); ax.grid(alpha=0.3)

    plt.tight_layout()
    pr_curve_path = os.path.join(run_dir, "pr_curve.png")
    plt.savefig(pr_curve_path, dpi=150)
    plt.show()
    print(f"PR curve saved to {pr_curve_path}")

    # ── 7. Final results ──────────────────────────────────────────────────────
    val_y_true, val_proba = get_predictions(model, loader_val)
    val_recall = recall_score(
        val_y_true, (val_proba >= best_thresh).astype(int), zero_division=0,
    )

    print(f"\n{'=' * 50}")
    print(f"  Final Results at best threshold = {best_thresh:.3f}")
    print(f"{'=' * 50}")
    print(f"  Test PR-AUC  : {test_pr:.4f}")
    print(f"  Test F1      : {best_f1:.4f}")
    print(f"  Test Recall  : {best_recall:.4f}")
    print(f"  Test Precision: {best_precision:.4f}")
    print(f"  Val  Recall  : {val_recall:.4f}")

    # ── 8. Save final metrics JSON ─────────────────────────────────────────────
    final_metrics = {
        "seed": SEED,
        "best_config": {
            "lr":             lr,
            "batch_size":     batch_size,
            "hidden_dim":     best_config["hidden_dim"],
            "weight_decay":   best_config["weight_decay"],
            "la_k":           best_config["la_k"],
            "la_alpha":       best_config["la_alpha"],
            "sched_patience": best_config["sched_patience"],
            "sched_factor":   best_config["sched_factor"],
        },
        "val_pr_auc":   val_pr,
        "test_pr_auc":  test_pr,
        "best_thresh":  best_thresh,
        "test_f1":      best_f1,
        "test_recall":  best_recall,
        "test_precision": best_precision,
        "val_recall":   val_recall,
    }

    metrics_path = os.path.join(run_dir, "final_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(final_metrics, f, indent=2)
    print(f"\nFinal metrics saved to {metrics_path}")

    # ── 9. Confusion matrix + classification report ───────────────────────────
    plot_confusion_matrix(y_true, y_proba, best_thresh, run_dir)

    # ── 10. Save standalone inference model ───────────────────────────────────
    # Self-contained: includes weights, config, threshold, and column metadata
    model_path = os.path.join(run_dir, "nn_model.pt")
    torch.save({
        "model_state_dict": best_state,
        "best_config": {
            "lr":             lr,
            "batch_size":     batch_size,
            "hidden_dim":     best_config["hidden_dim"],
            "weight_decay":   best_config["weight_decay"],
            "la_k":           best_config["la_k"],
            "la_alpha":       best_config["la_alpha"],
            "sched_patience": best_config["sched_patience"],
            "sched_factor":   best_config["sched_factor"],
        },
        "val_pr_auc":   val_pr,
        "test_pr_auc":  test_pr,
        "best_thresh":  best_thresh,
        "test_recall":  best_recall,
        "val_recall":   val_recall,
        "cat_cols":     cat_cols,
        "num_cols":     num_cols,
        "vocab_sizes":  vocab_sizes,
        "seed":         SEED,
    }, model_path)
    print(f"Model saved to {model_path}")

    print(f"\n{'=' * 50}")
    print(f"  All outputs saved to: {run_dir}/")
    print(f"    best.pt")
    print(f"    last.pt")
    print(f"    nn_model.pt")
    print(f"    nn_training_curves.png")
    print(f"    pr_curve.png")
    print(f"    confusion_matrix.png")
    print(f"    classification_report.txt")
    print(f"    training_log.csv")
    print(f"    hp_search_results.csv")
    print(f"    final_metrics.json")
    print(f"{'=' * 50}")


if __name__ == "__main__":
    main()
